# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnuragPanda700/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!pip -q install duckdb datasets huggingface_hub pyarrow

## 1. Unit of analysis + time window
### Unit of Analysis

One row represents the daily performance of one pseudonymized content item for one pseudonymized client on one report date.

### Time Window

This notebook analyzes the March 2026 partition (month = 2026-03), covering data from 2026-03-01 to 2026-03-31.

The verification queries confirmed that this partition contains 9,841,378 rows.

In [17]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected successfully!")

Connected successfully!


In [19]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/dim_clients.parquet')
""")
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/dim_clients.parquet')
LIMIT 5
""")

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

In [20]:
con.sql("""
SHOW TABLES
""")

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ 0 rows  │
└─────────┘

In [21]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [22]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

## 2. Fields: feature / label / context / excluded
### Feature

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

These fields are available before any future prediction and can be used as model features.

### Label

No prediction label is created in this notebook. This notebook focuses on defining and verifying the data contract.

### Context

- report_date
- client_hash_id
- content_hash_id
- month

These fields identify rows and provide context but should not be used directly as model features.

### Excluded

- client_hash_id (identifier only)
- content_hash_id (identifier only)
- Future information or label-derived fields (to prevent data leakage)

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected successfully!")

Connected successfully!


In [3]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │ start_date │  end_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

In [4]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [5]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         364347 │
└────────────────┘

In [6]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────┐
│ report_date │ client_hash_id │ content_hash_id │ duplicates │
│    date     │    varchar     │     varchar     │   int64    │
├─────────────┴────────────────┴─────────────────┴────────────┤
│                           0 rows                            │
└─────────────────────────────────────────────────────────────┘

In [7]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position,
    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_sessions
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┬──────────────────┐
│ total_rows │ missing_position │ missing_sessions │
│   int64    │      int128      │      int128      │
├────────────┼──────────────────┼──────────────────┤
│    9841378 │          6230317 │          3018741 │
└────────────┴──────────────────┴──────────────────┘

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)
### Verification Summary

The following queries verify the data contract.

- Grain verification returned zero duplicate rows.
- The March 2026 partition contains 9,841,378 rows.
- The available date range is 2026-03-01 to 2026-03-31.
- Availability was checked using gsc_data_available IS TRUE and ga4_data_available IS TRUE.
- Missing values were measured for important fields before any feature engineering.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits
### Data Limitations

This dataset has different history lengths for different clients.

Some rows do not contain Google Search Console or Google Analytics data because those services were not available for every client.

The final month of the panel should not be used for future label development because it represents the natural outcome window.

This dataset supports analysis and machine learning but cannot establish causal relationships.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.